# Comparaison de Performance : Solveur Spectral Signé pour le Problème 3-SAT

Ce notebook implémente un solveur spectral signé pour le problème 3-SAT.
L'approche consiste à éliminer complètement la partie orientée du Hamiltonien combinatoire. Toutes les clauses (1, 2, et 3-SAT) sont traduites en couplages quadratiques (arêtes signées) sur les variables originales, le nœud de référence T et des variables auxiliaires.

Les variables auxiliaires partageant une même paire orientée de littéraux sont contraintes par une arête de poids infini pour forcer leur fusion spectrale dans le Laplacien signé.

In [ ]:
import time
import random
import urllib.request
import numpy as np
import torch
import matplotlib.pyplot as plt
from pysat.solvers import Glucose4

print("GPU Disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nom du GPU :", torch.cuda.get_device_name(0))

In [ ]:
#@title Configuration de l'Instance { run: "auto" }
source_du_benchmark = "Circuit Miter (Aléatoire)" #@param ["SAT Competition (Industrial)", "Circuit Miter (Aléatoire)", "SATLIB Bounded Model Checking (BMC)", "SATLIB Planning (Blocks World)"]
sat_competition_instance = "Arithmetic Adder (add32)" #@param ["Arithmetic Adder (add32)", "Arithmetic Adder Large (add64)", "Arithmetic Adder Huge (add128)", "Logic Equivalence Miter (miter1)", "Factorization (prime169)", "Factorization Large (prime841)", "Square Root (sqrt10609)", "Square Root Large (sqrt63001)", "XOR Chain (xor6)", "Pigeonhole (ph6)"]
difficulte = "Facile" #@param ["Facile", "Moyen", "Difficile", "Très difficile"]
miter_nombre_portes = 150 #@param {type:"integer"}

COMPETITION_BENCHMARKS = {
    "Arithmetic Adder (add32)": "https://raw.githubusercontent.com/arminbiere/kissat/master/test/cnf/add32.cnf",
    "Arithmetic Adder Large (add64)": "https://raw.githubusercontent.com/arminbiere/kissat/master/test/cnf/add64.cnf",
    "Arithmetic Adder Huge (add128)": "https://raw.githubusercontent.com/arminbiere/kissat/master/test/cnf/add128.cnf",
    "Logic Equivalence Miter (miter1)": "https://raw.githubusercontent.com/arminbiere/kissat/master/test/cnf/miter1.cnf",
    "Factorization (prime169)": "https://raw.githubusercontent.com/arminbiere/kissat/master/test/cnf/prime169.cnf",
    "Factorization Large (prime841)": "https://raw.githubusercontent.com/arminbiere/kissat/master/test/cnf/prime841.cnf",
    "Square Root (sqrt10609)": "https://raw.githubusercontent.com/arminbiere/kissat/master/test/cnf/sqrt10609.cnf",
    "Square Root Large (sqrt63001)": "https://raw.githubusercontent.com/arminbiere/kissat/master/test/cnf/sqrt63001.cnf",
    "XOR Chain (xor6)": "https://raw.githubusercontent.com/arminbiere/kissat/master/test/cnf/xor6.cnf",
    "Pigeonhole (ph6)": "https://raw.githubusercontent.com/arminbiere/kissat/master/test/cnf/ph6.cnf"
}
PUBLIC_BENCHMARKS = {
    "SATLIB Bounded Model Checking (BMC)": {
        "Facile": {
            "url": "http://www.cs.ubc.ca/~hoos/SATLIB/Benchmarks/SAT/BMC/queueinvar10.cnf",
            "name": "queueinvar10"
        },
        "Moyen": {
            "url": "http://www.cs.ubc.ca/~hoos/SATLIB/Benchmarks/SAT/BMC/barrel7.cnf",
            "name": "barrel7"
        },
        "Difficile": {
            "url": "http://www.cs.ubc.ca/~hoos/SATLIB/Benchmarks/SAT/BMC/barrel8.cnf",
            "name": "barrel8"
        }
    },
    "SATLIB Planning (Blocks World)": {
        "Facile": {
            "url": "http://www.cs.ubc.ca/~hoos/SATLIB/Benchmarks/SAT/PLANNING/BlocksWorld/bw_large.a.cnf",
            "name": "bw_large.a"
        },
        "Moyen": {
            "url": "http://www.cs.ubc.ca/~hoos/SATLIB/Benchmarks/SAT/PLANNING/BlocksWorld/bw_large.b.cnf",
            "name": "bw_large.b"
        },
        "Difficile": {
            "url": "http://www.cs.ubc.ca/~hoos/SATLIB/Benchmarks/SAT/PLANNING/BlocksWorld/bw_large.c.cnf",
            "name": "bw_large.c"
        }
    }
}

In [ ]:
def generate_miter_circuit_3sat(n_inputs=30, n_gates=100, diff="Facile"):
    clauses = []
    next_var = n_inputs + 1
    gates1 = []
    gates2 = []
    
    current_vars = list(range(1, n_inputs + 1))
    for _ in range(n_gates):
        gtype = random.choice(["AND", "OR", "XOR"])
        in1 = random.choice(current_vars)
        in2 = random.choice(current_vars)
        out = next_var
        next_var += 1
        gates1.append((gtype, out, in1, in2))
        current_vars.append(out)
    output1 = current_vars[-1]
    
    n_mutations = 1 if diff == "Facile" else (2 if diff == "Moyen" else 4)
    mutated_indices = set(random.sample(range(n_gates), min(n_mutations, n_gates)))
    
    var_offset = next_var - (n_inputs + 1)
    
    for i, (gtype, out, in1, in2) in enumerate(gates1):
        out2 = out + var_offset
        in1_2 = in1 if in1 <= n_inputs else in1 + var_offset
        in2_2 = in2 if in2 <= n_inputs else in2 + var_offset
        
        if i in mutated_indices:
            new_type = "OR" if gtype == "AND" else "AND"
            gates2.append((new_type, out2, in1_2, in2_2))
        else: 
            gates2.append((gtype, out2, in1_2, in2_2))
            
    output2 = current_vars[-1] + var_offset
    next_var = output2 + 1
    
    for g1, g2 in [(gates1, ""), (gates2, "_2")]:
        for gtype, out, in1, in2 in g1:
            if gtype == "AND":
                clauses.append([in1, -out])
                clauses.append([in2, -out])
                clauses.append([-in1, -in2, out])
            elif gtype == "OR":
                clauses.append([-in1, out])
                clauses.append([-in2, out])
                clauses.append([in1, in2, -out])
            elif gtype == "XOR":
                clauses.append([-in1, -in2, -out])
                clauses.append([in1, in2, -out])
                clauses.append([-in1, in2, out])
                clauses.append([in1, -in2, out])
                
    miter_out = next_var
    clauses.append([-output1, -output2, -miter_out])
    clauses.append([output1, output2, -miter_out])
    clauses.append([-output1, output2, miter_out])
    clauses.append([output1, -output2, miter_out])
    clauses.append([miter_out])
    
    return convert_to_3sat(miter_out, clauses)

def parse_dimacs(content):
    clauses = []
    num_vars = 0
    current_clause = []
    for line in content.split('\n'):
        line = line.strip()
        if not line or line.startswith('c'):
            continue
        if line.startswith('p cnf'):
            parts = line.split()
            num_vars = int(parts[2])
            continue
        for token in line.split():
            val = int(token)
            if val == 0:
                if current_clause:
                    clauses.append(current_clause)
                    current_clause = []
            else: 
                current_clause.append(val)
    if current_clause:
        clauses.append(current_clause)
    return num_vars, clauses

def convert_to_3sat(num_vars, clauses):
    final_cnf = []
    next_var = num_vars + 1
    for c in clauses:
        if len(c) <= 3:
            final_cnf.append(c)
        else:
            curr = c
            while len(curr) > 3:
                aux = next_var
                next_var += 1
                final_cnf.append([curr[0], curr[1], aux])
                curr = [-aux] + curr[2:]
            final_cnf.append(curr)
    return next_var - 1, final_cnf

def load_selected_benchmark():
    if source_du_benchmark == "Circuit Miter (Aléatoire)":
        n_inputs = max(5, miter_nombre_portes // 5)
        return generate_miter_circuit_3sat(n_inputs=n_inputs, n_gates=miter_nombre_portes, diff=difficulte)
    elif source_du_benchmark == "SAT Competition (Industrial)":
        url = COMPETITION_BENCHMARKS[sat_competition_instance]
        print(f"Téléchargement du benchmark SAT Competition (Industrial) : {sat_competition_instance}...")
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        try:
            with urllib.request.urlopen(req, timeout=10) as response:
                content = response.read().decode('utf-8')
            num_vars, clauses = parse_dimacs(content)
            return convert_to_3sat(num_vars, clauses)
        except Exception as e:
            print(f"\n[ERREUR] Impossible de télécharger le benchmark ({e}).")
            n_inputs = max(5, miter_nombre_portes // 5)
            return generate_miter_circuit_3sat(n_inputs=n_inputs, n_gates=miter_nombre_portes, diff=difficulte)
    else:
        diff_satlib = difficulte if difficulte in ["Facile", "Moyen", "Difficile"] else "Difficile"
        bench_info = PUBLIC_BENCHMARKS[source_du_benchmark][diff_satlib]
        url = bench_info["url"]
        print(f"Téléchargement du benchmark public SATLIB : {bench_info['name']}...")
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        try:
            with urllib.request.urlopen(req, timeout=10) as response:
                content = response.read().decode('utf-8')
            num_vars, clauses = parse_dimacs(content)
            return convert_to_3sat(num_vars, clauses)
        except Exception as e:
            print(f"\n[ERREUR] Impossible de télécharger le benchmark ({e}).")
            n_inputs = max(5, miter_nombre_portes // 5)
            return generate_miter_circuit_3sat(n_inputs=n_inputs, n_gates=miter_nombre_portes, diff=difficulte)

## 2. Pré-traitement

In [ ]:
def recursive_unit_propagation_and_reductions(num_vars, clauses, verbose=True):
    if verbose:
        print(f"[Preprocessing] Élimination unitaire récursive & littéraux purs sur {num_vars} variables...")
    
    fixed_literals = {}
    active_clauses = [list(c) for c in clauses]
    active_vars = set(range(1, num_vars + 1))
    fixed_empty_clauses = 0
    
    changed = True
    while changed:
        changed = False
        
        empty_clauses = sum(1 for c in active_clauses if len(c) == 0)
        if empty_clauses:
            fixed_empty_clauses += empty_clauses
            if verbose:
                print(f"  -> {empty_clauses} clause(s) vide(s) retirée(s).")
            active_clauses = [c for c in active_clauses if len(c) > 0]
        
        unit_counts = {}
        for c in active_clauses:
            if len(c) == 1:
                lit = c[0]
                unit_counts[lit] = unit_counts.get(lit, 0) + 1
                
        if unit_counts:
            var_unit_info = {}
            for lit, count in unit_counts.items():
                var = abs(lit)
                pol = 1 if lit > 0 else -1
                if var not in var_unit_info:
                    var_unit_info[var] = {1: 0, -1: 0}
                var_unit_info[var][pol] = count
                
            for var in sorted(list(var_unit_info.keys())):
                info = var_unit_info[var]
                pos_units = info[1]
                neg_units = info[-1]
                
                if pos_units >= neg_units:
                    l_val = 1
                    m = pos_units
                    opp_units = neg_units
                else:
                    l_val = -1
                    m = neg_units
                    opp_units = pos_units
                    
                opp_clauses_count = 0
                for c in active_clauses:
                    if len(c) >= 2:
                        for lit in c:
                            if abs(lit) == var:
                                pol = 1 if lit > 0 else -1
                                if pol != l_val:
                                    opp_clauses_count += 1
                                    break
                                    
                k = opp_clauses_count + opp_units
                
                if m >= k:
                    if verbose:
                        print(f"  -> Assignation forcée de x_{var} = {l_val} (m={m} >= k={k})")
                    fixed_literals[var] = l_val
                    active_vars.remove(var)
                    
                    new_active_clauses = []
                    for c in active_clauses:
                        satisfied = False
                        for lit in c:
                            if abs(lit) == var:
                                pol = 1 if lit > 0 else -1
                                if pol == l_val:
                                    satisfied = True
                                    break
                        if not satisfied:
                            new_clause = [lit for lit in c if abs(lit) != var]
                            new_active_clauses.append(new_clause)
                    active_clauses = new_active_clauses
                    changed = True
                    break
            if changed:
                continue
                
        pos_counts = {v: 0 for v in active_vars}
        neg_counts = {v: 0 for v in active_vars}
        for c in active_clauses:
            for lit in c:
                var = abs(lit)
                if var in active_vars:
                    if lit > 0:
                        pos_counts[var] += 1
                    else:
                        neg_counts[var] += 1
                        
        pure_vars = []
        for var in list(active_vars):
            pos = pos_counts[var]
            neg = neg_counts[var]
            if pos > 0 and neg == 0:
                pure_vars.append((var, 1))
            elif neg > 0 and pos == 0:
                pure_vars.append((var, -1))
            elif pos == 0 and neg == 0:
                pure_vars.append((var, 1))
                
        if pure_vars:
            changed = True
            for var, val in pure_vars:
                if verbose:
                    print(f"  -> Élimination du littéral pur/orphelin x_{var} = {val}")
                fixed_literals[var] = val
                active_vars.remove(var)
            
            new_active_clauses = []
            for c in active_clauses:
                satisfied = False
                for lit in c:
                    var = abs(lit)
                    val = 1 if lit > 0 else -1
                    if var in fixed_literals and fixed_literals[var] == val:
                        satisfied = True
                        break
                if not satisfied:
                    new_clause = [lit for lit in c if abs(lit) in active_vars]
                    new_active_clauses.append(new_clause)
            active_clauses = new_active_clauses
            
    if verbose:
        print(f"  -> Preprocessing terminé : {len(active_vars)} variables actives, {len(active_clauses)} clauses actives (fixées : {len(fixed_literals)}).")
        if fixed_empty_clauses > 0:
            print(f"  -> Clauses vides cumulées : {fixed_empty_clauses} (contribution constante +{fixed_empty_clauses} * u).")
            
    return active_vars, active_clauses, fixed_literals, fixed_empty_clauses

## 3. Construction du Graphe Signé Quadratique

In [ ]:
def build_signed_graph_for_3sat(num_vars, active_vars, active_clauses, u=1.0, verbose=True):
    """
    Construit le graphe pondéré signé complet représentant la formule SAT sans partie orientée.
    Le nœud de référence T est indexé par 0.
    Les variables actives d'origine sont indexées par 1 à N_red.
    Les variables auxiliaires créées pour les clauses ternaires sont indexées à partir de N_red + 1.
    """
    if verbose:
        print("[Graph Construction] Début de la construction du graphe signé quadratique...")
        
    var_list = sorted(list(active_vars))
    var_to_idx = {v: i + 1 for i, v in enumerate(var_list)}
    N_red = len(var_list)
    
    edges = []
    clause3_idx = 0
    clause3_aux_nodes = []
    clause3_literal_pairs = []
    
    for c in active_clauses:
        if len(c) == 1:
            # 1-Clause (Unitaire) : u * (1 - L1)
            # Edge (v1, T): weight u, sign pol1
            lit = c[0]
            v = abs(lit)
            idx = var_to_idx[v]
            pol = 1 if lit > 0 else -1
            edges.append((idx, 0, u, pol))
            
        elif len(c) == 2:
            # 2-Clause (Binaire) : u * (1 - L1*L2)
            # Edges:
            # - (v1, T) de signe pol1 et de poids u/2
            # - (v2, T) de signe pol2 et de poids u/2
            # - (v1, v2) de signe -pol1 * pol2 et de poids u/2
            v1, v2 = abs(c[0]), abs(c[1])
            idx1, idx2 = var_to_idx[v1], var_to_idx[v2]
            pol1 = 1 if c[0] > 0 else -1
            pol2 = 1 if c[1] > 0 else -1
            edges.append((idx1, 0, u / 2.0, pol1))
            edges.append((idx2, 0, u / 2.0, pol2))
            edges.append((idx1, idx2, u / 2.0, -pol1 * pol2))
            
        elif len(c) == 3:
            # 3-Clause (Ternaire) : u * (1 - L1*L2*L3)
            # Nous créons une variable auxiliaire s
            v1, v2, v3 = abs(c[0]), abs(c[1]), abs(c[2])
            idx1, idx2, idx3 = var_to_idx[v1], var_to_idx[v2], var_to_idx[v3]
            pol1 = 1 if c[0] > 0 else -1
            pol2 = 1 if c[1] > 0 else -1
            pol3 = 1 if c[2] > 0 else -1
            
            s = N_red + 1 + clause3_idx
            clause3_aux_nodes.append(s)
            
            # Enregistrement des paires orientées pour la mutualisation
            lits = [(idx1, pol1), (idx2, pol2), (idx3, pol3)]
            lits.sort(key=lambda x: x[0])
            pairs = [
                (lits[0][0], lits[0][1], lits[1][0], lits[1][1]),
                (lits[1][0], lits[1][1], lits[2][0], lits[2][1]),
                (lits[0][0], lits[0][1], lits[2][0], lits[2][1])
            ]
            clause3_literal_pairs.append(pairs)
            
            # 1. Trois arêtes variables-référence (v_i, T) de signe pol_i, poids u/2
            edges.append((idx1, 0, u / 2.0, pol1))
            edges.append((idx2, 0, u / 2.0, pol2))
            edges.append((idx3, 0, u / 2.0, pol3))
            
            # 2. Trois arêtes auxiliaire-variables (s, v_i) de signe pol_i, poids u/2
            edges.append((s, idx1, u / 2.0, pol1))
            edges.append((s, idx2, u / 2.0, pol2))
            edges.append((s, idx3, u / 2.0, pol3))
            
            # 3. Une arête auxiliaire-référence (s, T) antiferromagnétique (signe -1), poids u/2
            edges.append((s, 0, u / 2.0, -1))
            
            # 4. Trois arêtes quadratiques d'origine (v_i, v_j) de signe -pol_i * pol_j, poids u/2
            edges.append((idx1, idx2, u / 2.0, -pol1 * pol2))
            edges.append((idx2, idx3, u / 2.0, -pol2 * pol3))
            edges.append((idx1, idx3, u / 2.0, -pol1 * pol3))
            
            clause3_idx += 1
            
    # Mutualisation : Contraction des auxiliaires partageant la même paire orientée
    n_clause3 = len(clause3_aux_nodes)
    if verbose:
        print(f"  -> Analyse de {n_clause3} clauses ternaires pour la contraction des auxiliaires...")
    
    contraction_edges_count = 0
    for i in range(n_clause3):
        s_i = clause3_aux_nodes[i]
        pairs_i = set(clause3_literal_pairs[i])
        for j in range(i + 1, n_clause3):
            s_j = clause3_aux_nodes[j]
            pairs_j = set(clause3_literal_pairs[j])
            
            # Si elles partagent au moins une paire orientée commune, on contracte les auxiliaires
            if pairs_i.intersection(pairs_j):
                # Arête ferromagnétique (+) dure de poids 100000 * u
                edges.append((s_i, s_j, 100000.0 * u, 1))
                contraction_edges_count += 1
                
    if verbose and contraction_edges_count > 0:
        print(f"  -> Ajout de {contraction_edges_count} arêtes dures de contraction (+infini) entre auxiliaires.")
        
    total_nodes = N_red + 1 + n_clause3
    A = np.zeros((total_nodes, total_nodes), dtype=np.float32)
    
    # Sommation et compensation des arêtes le long de chaque paire
    for u_node, v_node, weight, sign in edges:
        val = sign * weight
        A[u_node, v_node] += val
        A[v_node, u_node] += val
        
    if verbose:
        print(f"  -> Graphe complet construit : {total_nodes} nœuds (T, {N_red} variables actives, {n_clause3} auxiliaires).")
        
    return A, var_to_idx

## 4. Résolution par Clustering Spectral Signé

In [ ]:
def solve_signed_spectral_clustering(A, var_to_idx, active_vars, fixed_literals, clauses, verbose=True):
    """
    Résout le problème de clustering signé à l'aide de la décomposition spectrale du Laplacien signé.
    Le calcul est effectué sur le GPU via PyTorch CUDA si disponible.
    """
    N = A.shape[0]
    N_red = len(active_vars)
    
    if verbose:
        print(f"[Spectral Solver] Construction du Laplacien signé sur {N} nœuds...")
        
    # Calcul de la matrice de degrés signés D
    D = np.diag(np.sum(np.abs(A), axis=1))
    L_signed = D - A
    
    t_start = time.time()
    # Résolution de l'eigenproblem
    if torch.cuda.is_available():
        if verbose:
            print("  -> Résolution sur GPU (PyTorch CUDA)...")
        L_cuda = torch.tensor(L_signed, dtype=torch.float32, device='cuda')
        eigenvalues, eigenvectors = torch.linalg.eigh(L_cuda)
        
        v_min = eigenvectors[:, 0].cpu().numpy()
        eigenval = eigenvalues[0].item()
    else:
        if verbose:
            print("  -> Résolution sur CPU (scipy/numpy)...")
        eigenvalues, eigenvectors = np.linalg.eigh(L_signed)
        v_min = eigenvectors[:, 0]
        eigenval = eigenvalues[0]
        
    t_solve = time.time() - t_start
    if verbose:
        print(f"  -> Étape de résolution terminée en {t_solve:.4f}s. Plus petite valeur propre = {eigenval:.6f}")
        
    # Alignement du vecteur propre par rapport au spin de référence T (nœud 0)
    T_val = v_min[0]
    T_sign = np.sign(T_val) if abs(T_val) > 1e-9 else 1.0
    aligned_v = v_min * T_sign
    
    # Extraction des configurations pour les variables actives (nœuds 1 à N_red)
    var_list = sorted(list(active_vars))
    spins_red = np.sign(aligned_v)
    spins_red[spins_red == 0] = 1.0 # Par défaut si 0
    
    # Reconstruction des deux configurations candidates
    cand1 = {}
    cand2 = {}
    
    for idx_in_list, var in enumerate(var_list):
        node_idx = var_to_idx[var]
        # Configuration conforme à la réduction spectrale
        cand1[var] = int(spins_red[node_idx])
        # Configuration opposée
        cand2[var] = -int(spins_red[node_idx])
        
    # Restauration des variables fixées au prétraitement
    for var, val in fixed_literals.items():
        cand1[var] = val
        cand2[var] = val
        
    # Évaluation des énergies (nombre de clauses insatisfaites)
    def evaluate_energy(spins):
        unsat = 0
        for c in clauses:
            sat = False
            for lit in c:
                val = spins.get(abs(lit), 1)
                pol = 1 if lit > 0 else -1
                if val == pol:
                    sat = True
                    break
            if not sat:
                unsat += 1
        return unsat
        
    unsat1 = evaluate_energy(cand1)
    unsat2 = evaluate_energy(cand2)
    
    if verbose:
        print(f"  -> Candidat 1 (Spectral) : {unsat1} clauses insatisfaites.")
        print(f"  -> Candidat 2 (Opposé)   : {unsat2} clauses insatisfaites.")
        
    if unsat1 < unsat2:
        best_spins = cand1
        best_unsat = unsat1
    else:
        best_spins = cand2
        best_unsat = unsat2
        
    return best_spins, best_unsat, t_solve

## 5. Baseline CDCL (PySAT Glucose4)

In [ ]:
def solve_3sat_with_pysat(num_vars, clauses, verbose=True):
    t_start = time.time()
    if verbose:
        print("[CPU PySAT] Lancement de la baseline Glucose4...")
    with Glucose4(bootstrap_with=clauses) as solver:
        satisfied = solver.solve()
        model = solver.get_model() if satisfied else None
        
    if model:
        t_elapsed = time.time() - t_start
        if verbose:
            print(f"  -> Glucose4 a résolu le problème (SAT) en {t_elapsed:.4f}s.")
        spins = {abs(lit): (1 if lit > 0 else -1) for lit in model}
        return spins, t_elapsed, 0
    else:
        t_elapsed = time.time() - t_start
        if verbose:
            print(f"  -> Glucose4 a conclu à UNSAT en {t_elapsed:.4f}s.")
        default_spins = {v: 1 for v in range(1, num_vars + 1)}
        def evaluate_energy(spins):
            unsat = 0
            for c in clauses:
                sat = False
                for lit in c:
                    val = spins.get(abs(lit), 1)
                    pol = 1 if lit > 0 else -1
                    if val == pol:
                        sat = True
                        break
                if not sat:
                    unsat += 1
            return unsat
        return default_spins, t_elapsed, evaluate_energy(default_spins)

## 6. Orchestration et Évaluation

In [ ]:
def solve_3sat_spectral_only(num_vars, clauses, u=1.0, verbose=True):
    t_start = time.time()
    
    # 1. Preprocessing récursif
    active_vars, active_clauses, fixed_lits, empty_count = recursive_unit_propagation_and_reductions(num_vars, clauses, verbose=verbose)
    
    if len(active_vars) == 0:
        if verbose:
            print("[Orchestration] Toutes les variables ont été résolues lors du pré-traitement !")
        best_spins = {v: fixed_lits.get(v, 1) for v in range(1, num_vars + 1)}
        return best_spins, time.time() - t_start, empty_count
        
    # 2. Construction du graphe signé sans partie orientée
    A, var_to_idx = build_signed_graph_for_3sat(num_vars, active_vars, active_clauses, u=u, verbose=verbose)
    
    # 3. Clustering Spectral Signé
    best_spins, best_unsat, t_solve = solve_signed_spectral_clustering(A, var_to_idx, active_vars, fixed_lits, clauses, verbose=verbose)
    
    total_unsat = best_unsat + empty_count
    t_elapsed = time.time() - t_start
    
    if verbose:
        print(f"[Orchestration] Terminement en {t_elapsed:.4f}s. Clauses insatisfaites totales : {total_unsat} (dont {empty_count} vides).")
        
    return best_spins, t_elapsed, total_unsat

In [ ]:
verbose = True

num_vars, clauses_3sat = load_selected_benchmark()
print(f"\n--- Évaluation : {num_vars} variables, {len(clauses_3sat)} clauses ---")

# 1. Notre solveur spectral signé
print("\n[Exécution] Solveur Spectral Signé (GPU/CPU)...")
spectral_spins, spectral_time, spectral_unsat = solve_3sat_spectral_only(num_vars, clauses_3sat, u=1.0, verbose=verbose)

# 2. Baseline PySAT Glucose4
print("\n[Exécution] Baseline CDCL Glucose4...")
cpu_spins, cpu_time, cpu_unsat = solve_3sat_with_pysat(num_vars, clauses_3sat, verbose=verbose)

# Comparaison finale
print("\n=== BILAN COMPARATIF ===")
print(f"Solveur Spectral Signé : Clauses insatisfaites = {spectral_unsat} | Temps = {spectral_time:.4f} s")
print(f"Baseline CDCL Glucose  : Clauses insatisfaites = {cpu_unsat} | Temps = {cpu_time:.4f} s")